# LangChain Tutorial (using Qwen2.5 locally)

Hands-on guide to building LLM applications with [LangChain](https://python.langchain.com/) and a **local** Qwen2.5-7B model served by Ollama.

**Requirements**: `pip install langchain langchain-ollama langchain-community chromadb`
**LLM**: Qwen2.5-7B served locally with Ollama — no account, no token, fully offline.

> No data leaves your machine. The Ollama server runs at `http://localhost:11434`.

Pull the model before running:
```bash
ollama pull qwen2.5:7b
```

**Covered topics**:
1. Chat Models
2. Prompt Templates
3. Output Parsers
4. LCEL Chains
5. RAG Pipeline
6. Conversation Memory
7. Agents & Tools

In [1]:
!pip install -q langchain langchain-ollama langchain-community chromadb


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\julie\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Chat Models

LangChain's `ChatOllama` connects to your local Ollama server. It wraps the model
behind a unified interface so you can swap providers without changing application code.

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0,  # deterministic outputs for reproducibility
)

# Simple invocation
response = llm.invoke("What is LangChain in one sentence?")
print(response.content)

C:\Users\julie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LangChain is a framework that enables the integration of large language models with various external APIs and data sources to create more versatile and powerful applications.


In [3]:
from langchain_core.messages import SystemMessage, HumanMessage

# Using structured messages
messages = [
    SystemMessage(content="You are a concise Python tutor. Answer in 2-3 sentences max."),
    HumanMessage(content="Explain list comprehensions."),
]

response = llm.invoke(messages)
print(response.content)

List comprehensions provide a concise way to create lists based on existing lists or other iterables. They consist of brackets containing an expression followed by a `for` statement, then zero or more `for` or `if` statements. For example: `[x**2 for x in range(5)]` generates `[0, 1, 4, 9, 16]`.


In [4]:
# Streaming — tokens arrive one at a time
for chunk in llm.stream("Name 3 benefits of local LLM inference."):
    print(chunk.content, end="", flush=True)
print()  # trailing newline

Local Large Language Model (LLM) inference offers several benefits, including:

1. **Privacy and Security**: When you run an LLM locally on your device or server, the data remains within your control, reducing the risk of sensitive information being exposed to external parties. This is particularly important for handling personal or confidential data.

2. **Latency Reduction**: Local inference can significantly reduce latency compared to sending requests to a remote server. Since there's no need to transmit data over the network, responses are faster and more responsive, which is crucial for real-time applications like chatbots or interactive voice assistants.

3. **Offline Capabilities**: With local LLMs, you don't rely on an internet connection to use the model. This makes it possible to operate in environments where connectivity might be unreliable or non-existent, ensuring that your application can still function as intended without interruptions.

These benefits highlight why loca

## 2. Prompt Templates

Templates let you define reusable prompt skeletons with placeholders.
`ChatPromptTemplate` produces a list of messages ready to send to the model.

In [5]:
from langchain_core.prompts import ChatPromptTemplate

# Simple template with one variable
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Be concise."),
    ("human", "{question}"),
])

# Format and inspect the messages
messages = prompt.invoke({"domain": "machine learning", "question": "What is overfitting?"})
print(messages.to_messages())

[SystemMessage(content='You are an expert in machine learning. Be concise.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is overfitting?', additional_kwargs={}, response_metadata={})]


In [6]:
# Plug the template directly into the model
chain = prompt | llm
response = chain.invoke({"domain": "machine learning", "question": "What is overfitting?"})
print(response.content)

Overfitting occurs when a model learns the training data too well, capturing noise and details that do not generalize to new data. This results in poor performance on unseen data.


In [7]:
# Few-shot prompting
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate English to French."),
    ("human", "Hello"),
    ("ai", "Bonjour"),
    ("human", "How are you?"),
    ("ai", "Comment allez-vous ?"),
    ("human", "{input}"),
])

chain = few_shot_prompt | llm
response = chain.invoke({"input": "Good evening, my friend."})
print(response.content)

Bonsoir, mon ami.


## 3. Output Parsers

Output parsers transform raw model output into structured data.
LangChain includes parsers for strings, JSON, Pydantic models, and more.

In [8]:
from langchain_core.output_parsers import StrOutputParser

# StrOutputParser extracts just the text content from an AIMessage
chain = prompt | llm | StrOutputParser()

# Now returns a plain string instead of an AIMessage object
result = chain.invoke({"domain": "databases", "question": "What is an index?"})
print(type(result))  # <class 'str'>
print(result)

<class 'langchain_core.messages.base.TextAccessor'>
An index in a database is a data structure that improves the speed of data retrieval operations on a table. It allows for quicker access to rows based on columns whose values are used for filtering. Think of it as a guide or a map that helps you find your way quickly to specific information, much like an index in the back of a book helps you find specific topics.


In [9]:
from langchain_core.output_parsers import JsonOutputParser

json_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Always respond with valid JSON. "
     "No markdown, no code fences, just raw JSON."),
    ("human", "Give me a JSON object with the name, year, and language of the "
              "programming language: {language}"),
])

chain = json_prompt | llm | JsonOutputParser()
result = chain.invoke({"language": "Python"})
print(type(result))  # <class 'dict'>
print(result)

<class 'dict'>
{'name': 'Python', 'year': 1991, 'language': 'interpreted'}


In [10]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class MovieReview(BaseModel):
    title: str = Field(description="Movie title")
    rating: float = Field(description="Rating from 0 to 10")
    summary: str = Field(description="One-sentence summary")


parser = PydanticOutputParser(pydantic_object=MovieReview)

pydantic_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a movie critic. Respond ONLY with the requested JSON format.\n"
     "{format_instructions}"),
    ("human", "Review the movie: {movie}"),
])

chain = pydantic_prompt | llm | parser
review = chain.invoke({
    "movie": "Inception",
    "format_instructions": parser.get_format_instructions(),
})
print(type(review))  # <class 'MovieReview'>
print(f"Title: {review.title}")
print(f"Rating: {review.rating}")
print(f"Summary: {review.summary}")

<class '__main__.MovieReview'>
Title: Inception
Rating: 8.8
Summary: A complex heist film that explores dreams within dreams to steal an idea.


## 4. LCEL Chains

The **LangChain Expression Language** (LCEL) lets you compose components with the `|` operator.
Every component is a `Runnable` — you can pipe, branch, and parallelize them.

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# A simple chain: prompt -> model -> string output
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following text in one sentence."),
    ("human", "{text}"),
])

summarize_chain = summarize_prompt | llm | StrOutputParser()

result = summarize_chain.invoke({
    "text": (
        "LangChain is a framework for developing applications powered by "
        "large language models. It provides tools to connect LLMs with "
        "external data sources and enables building complex reasoning chains."
    )
})
print(result)

LangChain is a framework that connects large language models with external data sources to build complex reasoning applications.


In [12]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# RunnableParallel: run multiple chains at once on the same input
pros_prompt = ChatPromptTemplate.from_messages([
    ("system", "List 3 pros of the given technology. Be concise."),
    ("human", "{tech}"),
])
cons_prompt = ChatPromptTemplate.from_messages([
    ("system", "List 3 cons of the given technology. Be concise."),
    ("human", "{tech}"),
])

pros_chain = pros_prompt | llm | StrOutputParser()
cons_chain = cons_prompt | llm | StrOutputParser()

parallel_chain = RunnableParallel(pros=pros_chain, cons=cons_chain)
result = parallel_chain.invoke({"tech": "Kubernetes"})

print("=== Pros ===")
print(result["pros"])
print("\n=== Cons ===")
print(result["cons"])

=== Pros ===
1. Automated Deployment
2. Scalability
3. High Availability

=== Cons ===
1. Complexity in Setup and Management
2. Resource Intensive for Small Workloads
3. Steep Learning Curve


In [13]:
# Multi-step chain: generate code, then explain it
code_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a short Python function. Code only, no explanations."),
    ("human", "{task}"),
])

explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "Explain the following code in plain English. 2-3 sentences."),
    ("human", "{code}"),
])

# Chain step 1 output into step 2
code_chain = code_prompt | llm | StrOutputParser()
full_chain = (
    {"code": code_chain}  # capture code output
    | explain_prompt
    | llm
    | StrOutputParser()
)

result = full_chain.invoke({"task": "Fibonacci sequence generator"})
print(result)

This Python function generates the first `n` numbers of the Fibonacci sequence. It starts with a list containing the first two numbers, 0 and 1, then iteratively calculates the next number as the sum of the previous two until it has calculated `n` numbers.


## 5. RAG Pipeline

Build a simple **Retrieval-Augmented Generation** pipeline:
1. Create documents and split them into chunks
2. Embed chunks and store in a vector database (Chroma)
3. Retrieve relevant chunks and generate an answer

In [14]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Sample knowledge base
documents = [
    Document(
        page_content=(
            "Ollama is a tool for running large language models locally. "
            "It supports models like Llama, Qwen, Mistral, and Gemma. "
            "Ollama provides an OpenAI-compatible API at localhost:11434. "
            "Models are downloaded and managed automatically."
        ),
        metadata={"source": "ollama-docs"},
    ),
    Document(
        page_content=(
            "Qwen2.5 is a series of large language models developed by Alibaba. "
            "The 7B variant offers a good balance of performance and efficiency. "
            "It supports multiple languages and excels at reasoning tasks. "
            "Qwen2.5 uses a transformer architecture with grouped-query attention."
        ),
        metadata={"source": "qwen-docs"},
    ),
    Document(
        page_content=(
            "LangChain is a framework for building applications with LLMs. "
            "It provides abstractions for prompts, chains, memory, and agents. "
            "LCEL (LangChain Expression Language) uses the pipe operator for composition. "
            "LangChain integrates with vector stores, document loaders, and tools."
        ),
        metadata={"source": "langchain-docs"},
    ),
    Document(
        page_content=(
            "ChromaDB is an open-source vector database for AI applications. "
            "It stores embeddings and supports similarity search. "
            "Chroma can run in-memory or persist to disk. "
            "It integrates natively with LangChain and other AI frameworks."
        ),
        metadata={"source": "chroma-docs"},
    ),
]

# Split into smaller chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: {chunk.page_content[:80]}...")

Split 4 documents into 8 chunks
  Chunk 0: Ollama is a tool for running large language models locally. It supports models l...
  Chunk 1: Models are downloaded and managed automatically....
  Chunk 2: Qwen2.5 is a series of large language models developed by Alibaba. The 7B varian...
  Chunk 3: excels at reasoning tasks. Qwen2.5 uses a transformer architecture with grouped-...
  Chunk 4: LangChain is a framework for building applications with LLMs. It provides abstra...
  Chunk 5: uses the pipe operator for composition. LangChain integrates with vector stores,...
  Chunk 6: ChromaDB is an open-source vector database for AI applications. It stores embedd...
  Chunk 7: natively with LangChain and other AI frameworks....


In [15]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

# Embed chunks and store in Chroma (in-memory)
embeddings = OllamaEmbeddings(model="qwen2.5:7b")
vectorstore = Chroma.from_documents(chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Test retrieval
results = retriever.invoke("What models does Ollama support?")
print(f"Retrieved {len(results)} chunks:")
for doc in results:
    print(f"  [{doc.metadata['source']}] {doc.page_content[:80]}...")

Retrieved 3 chunks:
  [ollama-docs] Models are downloaded and managed automatically....
  [chroma-docs] natively with LangChain and other AI frameworks....
  [qwen-docs] Qwen2.5 is a series of large language models developed by Alibaba. The 7B varian...


In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs):
    """Join retrieved documents into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)


rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question based only on the following context. "
     "If the context doesn't contain the answer, say so.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Ask questions against our knowledge base
answer = rag_chain.invoke("What architecture does Qwen2.5 use?")
print(f"Q: What architecture does Qwen2.5 use?")
print(f"A: {answer}")

print()

answer = rag_chain.invoke("How does ChromaDB store data?")
print(f"Q: How does ChromaDB store data?")
print(f"A: {answer}")

Q: What architecture does Qwen2.5 use?
A: The given context does not provide information about the specific architecture used by Qwen2.5. Therefore, I cannot answer the question based on the provided information.

Q: How does ChromaDB store data?
A: The provided context does not contain any information about how ChromaDB stores data. Therefore, I cannot answer the question based on the given information.


## 6. Conversation Memory

Add chat history so the model remembers previous turns.
LangChain uses `RunnableWithMessageHistory` to inject history into chains automatically.

In [17]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

# Prompt with a placeholder for chat history
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = memory_prompt | llm | StrOutputParser()

# Store sessions in a dict (in production, use a persistent store)
session_store: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]


chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "demo-session"}}

# Turn 1
r1 = chain_with_history.invoke({"input": "My name is Alice. I work on NLP."}, config=config)
print(f"Turn 1: {r1}")

# Turn 2 — the model should remember the name and topic
r2 = chain_with_history.invoke({"input": "What's my name and what do I work on?"}, config=config)
print(f"Turn 2: {r2}")

# Inspect stored history
print(f"\nHistory length: {len(session_store['demo-session'].messages)} messages")

Turn 1: Nice to meet you, Alice! As an NLP specialist, you likely deal with tasks like text analysis, sentiment classification, and language generation. How can I assist you further?
Turn 2: Your name is Alice, and you work on NLP (Natural Language Processing).

History length: 4 messages


## 7. Agents & Tools

Agents let the LLM decide **which tools to call** and in what order.
We define tools as Python functions and let the agent orchestrate them.

In [18]:
from langchain_core.tools import tool
import math


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b


@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b


@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number."""
    return math.sqrt(x)


tools = [multiply, add, square_root]
print("Available tools:")
for t in tools:
    print(f"  - {t.name}: {t.description}")

Available tools:
  - multiply: Multiply two numbers together.
  - add: Add two numbers together.
  - square_root: Calculate the square root of a number.


In [19]:
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

# Create agent LLM (need a fresh instance to bind tools)
agent_llm = ChatOllama(model="qwen2.5:7b", temperature=0)

# Create a ReAct agent using LangGraph
agent = create_react_agent(agent_llm, tools)

# The agent decides which tools to call to answer the question
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the square root of 144 plus 8 times 3?"}]}
)

# Print the conversation
for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "content") and msg.content:
        print(f"[{role}] {msg.content}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[{role}] -> tool call: {tc['name']}({tc['args']})")

C:\Users\julie\AppData\Local\Temp\ipykernel_65424\1365184119.py:8: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(agent_llm, tools)


[HumanMessage] What is the square root of 144 plus 8 times 3?
[AIMessage] To solve this problem, we need to follow these steps:

1. Calculate \( \sqrt{144} \).
2. Multiply 8 by 3.
3. Add the results from steps 1 and 2.

Let's start with step 1: calculating the square root of 144.

[AIMessage] -> tool call: square_root({'x': 144})
[ToolMessage] 12.0
[AIMessage] The square root of 144 is 12.0. 

Next, we'll proceed to step 2 and calculate \(8 \times 3\).

[AIMessage] -> tool call: multiply({'a': 8, 'b': 3})
[ToolMessage] 24.0
[AIMessage] Step 2 has given us the result: \(8 \times 3 = 24.0\).

Finally, we need to add the results from steps 1 and 2 together:

\[ 12.0 + 24.0 \]

Let's perform this addition.

[AIMessage] -> tool call: add({'a': 12, 'b': 24})
[ToolMessage] 36.0
[AIMessage] The final result of adding the square root of 144 (which is 12.0) to 8 times 3 (which is 24.0) is \( 36.0 \).

So, the answer is 36.0.


In [20]:
# Agent with a more complex question requiring multiple tool calls
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Multiply 15 by 7, then add the square root of 49 to the result."}]}
)

for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "content") and msg.content:
        print(f"[{role}] {msg.content}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[{role}] -> tool call: {tc['name']}({tc['args']})")

[HumanMessage] Multiply 15 by 7, then add the square root of 49 to the result.
[AIMessage] First, I will multiply 15 by 7, and then we'll add the square root of 49 to that result.

Let's start with multiplying 15 by 7.

[AIMessage] -> tool call: multiply({'a': 15, 'b': 7})
[ToolMessage] 105.0
[AIMessage] The multiplication of 15 by 7 gives us 105. Now, let's add the square root of 49 to this result.

Next, I will calculate the square root of 49.

[AIMessage] -> tool call: square_root({'x': 49})
[ToolMessage] 7.0
[AIMessage] The square root of 49 is 7. Now, we'll add this to our previous result of 105.

So, let's perform the addition: 105 + 7.

[AIMessage] -> tool call: add({'a': 105, 'b': 7})
[ToolMessage] 112.0
[AIMessage] The final result after multiplying 15 by 7 and then adding the square root of 49 is 112.


In [21]:
# Cleanup: delete the in-memory vector store
vectorstore.delete_collection()
print("Done! Vector store cleaned up.")

Done! Vector store cleaned up.


## Summary

| Concept | Key Class / Function |
|---|---|
| Chat Models | `ChatOllama` |
| Prompt Templates | `ChatPromptTemplate`, `MessagesPlaceholder` |
| Output Parsers | `StrOutputParser`, `JsonOutputParser`, `PydanticOutputParser` |
| LCEL Chains | `\|` operator, `RunnableParallel`, `RunnablePassthrough` |
| RAG | `RecursiveCharacterTextSplitter`, `Chroma`, `OllamaEmbeddings` |
| Memory | `RunnableWithMessageHistory`, `InMemoryChatMessageHistory` |
| Agents | `@tool`, `create_react_agent` (LangGraph) |

All examples run **100% locally** with Qwen2.5-7B via Ollama.